# Part 2: Exploratory Data Analysis with Spark

**Objective:** Perform EDA on `bank.csv` using Apache Spark to uncover trends, patterns, and anomalies — mirroring how banks first explore their customer base before modeling.

This notebook runs the same logic as `spark/eda.py` and `spark/feature_engineering.py`, cell by cell, so the distributed execution and intermediate outputs are visible interactively.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = (
    SparkSession.builder
    .appName("BankEDA_Notebook")
    .master("local[*]")
    .config("spark.sql.shuffle.partitions", "8")
    .config("spark.driver.memory", "2g")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")
print("Spark version:", spark.version)

In [ ]:
df = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv("../data/bank.csv")
)
df = df.withColumnRenamed("default", "credit_default")
print(f"Rows: {df.count()}  |  Columns: {len(df.columns)}")
df.printSchema()

### 2.1 Data Quality Check — Nulls and 'unknown' markers

In [ ]:
null_counts = df.select([
    F.count(F.when(F.col(c).isNull() | (F.col(c) == ""), c)).alias(c)
    for c in df.columns
])
null_counts.show(truncate=False)

for col in ["job", "education", "contact", "poutcome"]:
    cnt = df.filter(F.col(col) == "unknown").count()
    print(f"  {col:15s}: {cnt} 'unknown' values")

dup_count = df.count() - df.dropDuplicates().count()
print(f"\nDuplicate rows: {dup_count}")

### 2.2 Target Variable Distribution — Class Imbalance

In [ ]:
target_dist = (
    df.groupBy("y")
    .agg(F.count("*").alias("count"))
    .withColumn("percentage", F.round(F.col("count") / df.count() * 100, 2))
    .orderBy("y")
)
target_dist.show()

yes_count = df.filter(F.col("y") == "yes").count()
no_count  = df.filter(F.col("y") == "no").count()
print(f"Class imbalance ratio no:yes = {no_count}:{yes_count} ({no_count/yes_count:.1f}x)")

**Finding:** 88.5% No vs 11.5% Yes — a 7.7× imbalance. This is why accuracy alone is a poor metric later (Part 3 uses ROC-AUC instead).

### 2.3 Numerical Feature Statistics and Outliers

In [ ]:
num_cols = ["age", "balance", "duration", "campaign", "pdays", "previous"]
df.select(num_cols).describe().show()

q1, q3 = df.approxQuantile("balance", [0.25, 0.75], 0.01)
iqr = q3 - q1
lower_fence, upper_fence = q1 - 1.5 * iqr, q3 + 1.5 * iqr
outliers = df.filter((F.col("balance") < lower_fence) | (F.col("balance") > upper_fence)).count()
print(f"Balance IQR range: [{lower_fence:.0f}, {upper_fence:.0f}]")
print(f"Outlier rows (balance): {outliers} ({outliers/df.count()*100:.1f}%)")

### 2.4 Subscription Rate by Key Categorical Features

In [ ]:
def sub_rate(group_col):
    return (
        df.groupBy(group_col)
        .agg(
            F.count("*").alias("total"),
            F.sum(F.when(F.col("y") == "yes", 1).otherwise(0)).alias("subscribed")
        )
        .withColumn("sub_rate_pct", F.round(F.col("subscribed") / F.col("total") * 100, 2))
        .orderBy(F.col("sub_rate_pct").desc())
    )

for col in ["job", "education", "marital", "poutcome", "month"]:
    print(f"\n--- Subscription rate by {col} ---")
    sub_rate(col).show(truncate=False)

### 2.5 Correlation Analysis

In [ ]:
for col in ["age", "duration", "campaign", "previous"]:
    corr = df.stat.corr("balance", col)
    print(f"balance <-> {col:12s}: {corr:.4f}")

**Finding:** Correlations with balance are weak across the board (|r| < 0.1) — balance behaves largely independently of these features, which is itself a useful insight (no strong multicollinearity risk for linear models).

### 2.6 Visualizations

Saved to `docs/plots/` by `spark/eda.py` (rendered here for reference):

- `01_target_distribution.png` — class imbalance bar chart
- `02_sub_rate_by_job.png` — horizontal bar chart, subscription rate by job
- `03_age_distribution.png` — overlapping histograms by subscription outcome
- `04_duration_by_subscription.png` — boxplot of call duration vs outcome
- `05_sub_rate_by_month.png` — subscription rate by month

In [ ]:
from IPython.display import Image, display
import os

plot_dir = "../docs/plots"
for fname in sorted(os.listdir(plot_dir)):
    if fname.endswith(".png"):
        print(fname)
        display(Image(filename=os.path.join(plot_dir, fname)))

---
## Feature Engineering

Continuing with the same Spark session, this section replicates `spark/feature_engineering.py` to prepare the data for ML.

In [ ]:
def get_mode(sdf, col_name):
    return sdf.groupBy(col_name).count().orderBy(F.col("count").desc()).first()[col_name]

cols_with_unknown = ["job", "education", "contact", "poutcome"]
for col in cols_with_unknown:
    mode_val = get_mode(df, col)
    df = df.withColumn(col, F.when(F.col(col) == "unknown", mode_val).otherwise(F.col(col)))
    print(f"Replaced 'unknown' in '{col}' with mode: '{mode_val}'")

In [ ]:
# pdays = -1 means never contacted before -> boolean flag + clamp to 0
df = (
    df
    .withColumn("was_contacted_before", F.when(F.col("pdays") == -1, 0).otherwise(1).cast("int"))
    .withColumn("pdays", F.when(F.col("pdays") == -1, 0).otherwise(F.col("pdays")))
)

# Domain-driven engineered features
median_balance = df.approxQuantile("balance", [0.5], 0.01)[0]
df = (
    df
    .withColumn("balance_per_age", F.round(F.col("balance") / F.col("age"), 2))
    .withColumn("is_high_balance", F.when(F.col("balance") > median_balance, 1).otherwise(0).cast("int"))
    .withColumn("contact_intensity", F.col("campaign") + F.col("previous"))
    .withColumn("is_long_call", F.when(F.col("duration") > 300, 1).otherwise(0).cast("int"))
    .withColumn("season",
        F.when(F.col("month").isin("dec","jan","feb"), "winter")
         .when(F.col("month").isin("mar","apr","may"), "spring")
         .when(F.col("month").isin("jun","jul","aug"), "summer")
         .otherwise("autumn"))
    .withColumn("has_any_loan", F.when((F.col("housing")=="yes")|(F.col("loan")=="yes"),1).otherwise(0).cast("int"))
)
df.select("balance_per_age","is_high_balance","contact_intensity","is_long_call","season","has_any_loan").show(5)

### 2.7 Encoding Pipeline (StringIndexer → OneHotEncoder → VectorAssembler → StandardScaler)

In [ ]:
from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler, StandardScaler

df = df.withColumn("label", F.when(F.col("y") == "yes", 1.0).otherwise(0.0))

categorical_cols = ["job","marital","education","credit_default","housing","loan","contact","month","poutcome","season"]
numerical_cols   = ["age","balance","duration","campaign","pdays","previous","day","balance_per_age","contact_intensity"]

indexers = [StringIndexer(inputCol=c, outputCol=f"{c}_idx", handleInvalid="keep") for c in categorical_cols]
encoder  = OneHotEncoder(inputCols=[f"{c}_idx" for c in categorical_cols], outputCols=[f"{c}_ohe" for c in categorical_cols])

all_feature_cols = [f"{c}_ohe" for c in categorical_cols] + numerical_cols + \
                    ["was_contacted_before","is_high_balance","is_long_call","has_any_loan"]

assembler = VectorAssembler(inputCols=all_feature_cols, outputCol="features_raw", handleInvalid="keep")
scaler    = StandardScaler(inputCol="features_raw", outputCol="features", withMean=True, withStd=True)

pipeline = Pipeline(stages=indexers + [encoder, assembler, scaler])
pipeline_model = pipeline.fit(df)
df_final = pipeline_model.transform(df)

print("Feature vector size:", df_final.select("features").first()[0].size)
df_final.select("label","features").show(5, truncate=True)

In [ ]:
train_df, test_df = df_final.randomSplit([0.8, 0.2], seed=42)
print(f"Train: {train_df.count()} rows | Test: {test_df.count()} rows")

train_df.select("label","features").write.mode("overwrite").parquet("../data/train.parquet")
test_df.select("label","features").write.mode("overwrite").parquet("../data/test.parquet")
pipeline_model.write().overwrite().save("../data/feature_pipeline_model")
print("Saved train/test parquet + feature_pipeline_model")

In [ ]:
spark.stop()
print("EDA + Feature Engineering complete")

### 2.8 Summary of Key Findings

| Insight | Finding |
|---------|---------|
| Class imbalance | 88.5% No vs 11.5% Yes |
| Strongest predictor | Call duration (>5 min → 33%+ conversion) |
| Best job segment | Retired (23.5%) & Students (22.6%) |
| Best months | Oct, Dec, Mar (30–46%) vs May (6.7%) |
| Previous success | 65% re-subscribe if previous outcome = success |
| Balance effect | High-balance customers subscribe ~2× more |
| Feature vector | 50-dimensional after one-hot encoding + scaling |

Next: **`03_model_training_validation_spark_ml.ipynb`** — training and validating ML models on this feature set.